In [9]:
%load_ext autoreload
%autoreload 2

import hashlib
import time
import re
import chromadb
from chromadb.config import Settings
from chromadb import Collection
from datetime import datetime
from pyprojroot import here
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from odyssey import OllamaEmbeddings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
BOOKS_DIR = here("data/books")

In [11]:
def extract_number_from_file_name(name: str) -> int:
    
    result = re.search(r"\(\d+\)", name)
    if result is None:
        raise ValueError()
    
    return int(result.group()[1:-1])
    

docs: list[Document] = []
for file in BOOKS_DIR.glob("*.txt"):
    book_num = extract_number_from_file_name(file.name)

    with open(file, "r") as fp:
        page_content = fp.read()
    
    docs.append(
        Document(
            page_content=page_content,
            metadata={
                "document_id": hashlib.sha256(page_content.encode("utf-8")).hexdigest(),
                "source": str(file),
                "book": page_content.splitlines()[0],
                "book_num": book_num
            }
        )
    )    

docs[:3]

[Document(metadata={'document_id': '9c834fb000aa5b4f80678fc85c36a9cbb7139fab5b12b739b1272e2cb026f08b', 'source': 'c:\\Projects\\odyssey\\data\\books\\book_i (1).txt', 'book': 'BOOK I', 'book_num': 1}, page_content='BOOK I\n\nTell me, O muse, of that ingenious hero who travelled far and wide\nafter he had sacked the famous town of Troy. Many cities did he visit,\nand many were the nations with whose manners and customs he was acquainted;\nmoreover he suffered much by sea while trying to save his own life\nand bring his men safely home; but do what he might he could not save\nhis men, for they perished through their own sheer folly in eating\nthe cattle of the Sun-god Hyperion; so the god prevented them from\never reaching home. Tell me, too, about all these things, O daughter\nof Jove, from whatsoever source you may know them. \n\nSo now all who escaped death in battle or by shipwreck had got safely\nhome except Ulysses, and he, though he was longing to return to his\nwife and country, 

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

splits = splitter.split_documents(docs)
print(f"Created {len(splits)} splits.")

splits[:3]

Created 896 splits.


[Document(metadata={'document_id': '9c834fb000aa5b4f80678fc85c36a9cbb7139fab5b12b739b1272e2cb026f08b', 'source': 'c:\\Projects\\odyssey\\data\\books\\book_i (1).txt', 'book': 'BOOK I', 'book_num': 1, 'start_index': 0}, page_content='BOOK I\n\nTell me, O muse, of that ingenious hero who travelled far and wide\nafter he had sacked the famous town of Troy. Many cities did he visit,\nand many were the nations with whose manners and customs he was acquainted;\nmoreover he suffered much by sea while trying to save his own life\nand bring his men safely home; but do what he might he could not save\nhis men, for they perished through their own sheer folly in eating\nthe cattle of the Sun-god Hyperion; so the god prevented them from\never reaching home. Tell me, too, about all these things, O daughter\nof Jove, from whatsoever source you may know them.'),
 Document(metadata={'document_id': '9c834fb000aa5b4f80678fc85c36a9cbb7139fab5b12b739b1272e2cb026f08b', 'source': 'c:\\Projects\\odyssey\\data

In [13]:
def hash_document(document: Document) -> str:

    content = (
        f"{document.metadata["source"]}-"
        f"{document.metadata["start_index"]}-"
        f"{document.page_content}"
    )

    return hashlib.sha256(content.encode("utf-8")).hexdigest()

for doc in splits:
    doc.id = hash_document(doc)

splits[0]

Document(id='e29b5f1658245c9b54b7c994a26f20a5c3a01b62e0b2921aebe9e7b98f9691f7', metadata={'document_id': '9c834fb000aa5b4f80678fc85c36a9cbb7139fab5b12b739b1272e2cb026f08b', 'source': 'c:\\Projects\\odyssey\\data\\books\\book_i (1).txt', 'book': 'BOOK I', 'book_num': 1, 'start_index': 0}, page_content='BOOK I\n\nTell me, O muse, of that ingenious hero who travelled far and wide\nafter he had sacked the famous town of Troy. Many cities did he visit,\nand many were the nations with whose manners and customs he was acquainted;\nmoreover he suffered much by sea while trying to save his own life\nand bring his men safely home; but do what he might he could not save\nhis men, for they perished through their own sheer folly in eating\nthe cattle of the Sun-god Hyperion; so the god prevented them from\never reaching home. Tell me, too, about all these things, O daughter\nof Jove, from whatsoever source you may know them.')

---

In [14]:
embed_model = "nomic-embed-text"
embed_fn = OllamaEmbeddings(model=embed_model)

In [15]:
client = chromadb.PersistentClient(path=here("db/chroma"), settings=Settings(allow_reset=True))
client.reset()

True

In [16]:
collection = client.create_collection(
    get_or_create=True,
    name="books",
    embedding_function=embed_fn,
    metadata={
        "embed_model": embed_model,
        "created_at": str(datetime.now())
    }
)

In [17]:
def add_in_batches(
        collection: Collection, 
        splits: list[Document], 
        batch_size: int=32, 
        retries: int=3
):
    total_batches = (len(splits) + batch_size - 1) // batch_size

    for batch_num, i in enumerate(range(0, len(splits), batch_size), start=1):
        batch = splits[i:i + batch_size]

        print(f"Adding batch {batch_num}/{total_batches} ({len(batch)} documents)")

        for attempt in range(1, retries + 1):
            try:
                collection.add(
                    ids=[x.id for x in batch],
                    documents=[x.page_content for x in batch],
                    metadatas=[x.metadata for x in batch],
                )

                break

            except Exception as e:
                print(
                    f"Batch {batch_num} failed "
                    f"(attempt {attempt}/{retries}): {e}"
                )

                if attempt == retries:
                    raise

                time.sleep(2 ** attempt)  # exponential backoff

In [18]:
add_in_batches(
    collection,
    splits,
    batch_size=32,
    retries=3,
)

Adding batch 1/28 (32 documents)
Adding batch 2/28 (32 documents)
Adding batch 3/28 (32 documents)
Adding batch 4/28 (32 documents)
Adding batch 5/28 (32 documents)
Adding batch 6/28 (32 documents)
Adding batch 7/28 (32 documents)
Adding batch 8/28 (32 documents)
Adding batch 9/28 (32 documents)
Adding batch 10/28 (32 documents)
Adding batch 11/28 (32 documents)
Adding batch 12/28 (32 documents)
Adding batch 13/28 (32 documents)
Adding batch 14/28 (32 documents)
Adding batch 15/28 (32 documents)
Adding batch 16/28 (32 documents)
Adding batch 17/28 (32 documents)
Adding batch 18/28 (32 documents)
Adding batch 19/28 (32 documents)
Adding batch 20/28 (32 documents)
Adding batch 21/28 (32 documents)
Adding batch 22/28 (32 documents)
Adding batch 23/28 (32 documents)
Adding batch 24/28 (32 documents)
Adding batch 25/28 (32 documents)
Adding batch 26/28 (32 documents)
Adding batch 27/28 (32 documents)
Adding batch 28/28 (32 documents)
